# Удаление шума с помощью свёрточного автоэнкодера

Обучаем convolutional denoising autoencoder восстанавливать чистые
изображения MNIST по зашумлённым. На вход модели подаётся изображение
с шумом, а целевой выход — соответствующий чистый оригинал.

**Цели:** понять кодирование и восстановление изображения, сравнить
ошибку до и после обработки и визуально проверить реконструкции.

Нужны NumPy, Matplotlib, scikit-learn и TensorFlow. Выполняйте ячейки
сверху вниз; первый запуск может скачивать MNIST.

## 1. Данные и воспроизводимость

Из стандартных 60 000 train-изображений выделяем 6 000 для validation.
Метки цифр используются только для стратификации; автоэнкодер обучается
без целевых меток классов. Стандартные 10 000 test-изображений оставляем
для итоговой оценки.

Изображения имеют форму `(28, 28, 1)`, тип `float32` и значения в `[0, 1]`.
Seed фиксирует разбиение, шум и обучение. Детерминированность TensorFlow
относится к одному окружению; версии библиотек и устройства могут влиять
на результат.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from sklearn.model_selection import train_test_split

seed = 42
noise_factor = 0.50
epochs = 1
batch_size = 128

tf.keras.utils.set_random_seed(seed)
tf.config.experimental.enable_op_determinism()
noise_rng = np.random.default_rng(seed)
plot_rng = np.random.default_rng(seed)

(x_train, y_train), (x_test, _) = mnist.load_data()
x_train = x_train.astype("float32").reshape(-1, 28, 28, 1)
x_train /= 255.0
x_test = x_test.astype("float32").reshape(-1, 28, 28, 1)
x_test /= 255.0
x_fit, x_val = train_test_split(
    x_train, test_size=6000, stratify=y_train, random_state=seed
)
del x_train, y_train

print("Train:", x_fit.shape)
print("Validation:", x_val.shape)
print("Test:", x_test.shape)

## 2. Добавление шума

К каждому пикселю добавляем нормальный шум с нулевым средним
и стандартным отклонением `0.50`, затем ограничиваем значения диапазоном `[0, 1]`:

$$\tilde{x} = \operatorname{clip}(x + \epsilon, 0, 1),\qquad
\epsilon \sim N(0, 0.50^2).$$

Генерируем шум сразу для массива изображений вместо Python-цикла.
Один фиксированный набор шума используется во всех эпохах. Для оценки
и рисунков используется один и тот же зашумлённый test-набор.

In [ ]:
def add_noise(images, noise_factor, rng):
    noise = rng.normal(0, noise_factor, size=images.shape).astype("float32")
    return np.clip(images + noise, 0, 1)


x_fit_noisy = add_noise(x_fit, noise_factor, noise_rng)
x_val_noisy = add_noise(x_val, noise_factor, noise_rng)
x_test_noisy = add_noise(x_test, noise_factor, noise_rng)

## 3. Архитектура и обучение

Свёртки с 32, 64 и 128 фильтрами
сжимают изображение до 7 × 7, затем создаётся код из 64 признаков.
Декодер разворачивает код и увеличивает размер с помощью транспонированных
свёрток. Sigmoid ограничивает выход диапазоном `[0, 1]`.

Функция потерь — MSE между чистым изображением и выходом модели.
Используем Adam, одну эпоху и batch size 128. Одна эпоха —
учебный бюджет, а не заявление о достигнутой сходимости.

In [ ]:
def build_convolutional_autoencoder():
    inputs = layers.Input(shape=(28, 28, 1))
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(inputs)
    x = layers.Conv2D(
        64, (3, 3), activation="relu", padding="same", strides=(2, 2)
    )(x)
    x = layers.Conv2D(
        128, (3, 3), activation="relu", padding="same", strides=(2, 2)
    )(x)
    x = layers.Flatten()(x)
    encoded = layers.Dense(64, activation="relu")(x)

    x = layers.Dense(7 * 7 * 128, activation="relu")(encoded)
    x = layers.Reshape((7, 7, 128))(x)
    x = layers.Conv2DTranspose(
        64, (3, 3), activation="relu", padding="same", strides=(2, 2)
    )(x)
    x = layers.Conv2DTranspose(
        32, (3, 3), activation="relu", padding="same", strides=(2, 2)
    )(x)
    decoded = layers.Conv2DTranspose(
        1, (3, 3), activation="sigmoid", padding="same"
    )(x)
    return models.Model(inputs, decoded, name="denoising_autoencoder")


autoencoder = build_convolutional_autoencoder()
autoencoder.compile(optimizer="adam", loss="mean_squared_error")
autoencoder.summary()

In [ ]:
history = autoencoder.fit(
    x_fit_noisy, x_fit, epochs=epochs, batch_size=batch_size,
    validation_data=(x_val_noisy, x_val), shuffle=True,
    verbose=2,
)

## 4. MSE до и после восстановления

Для каждого test-изображения считаем две ошибки, усредняя по всем пикселям:

$$\operatorname{MSE}_{\mathrm{noise}} = \operatorname{mean}\bigl[(x - \tilde{x})^2\bigr],$$

$$\operatorname{MSE}_{\mathrm{reconstruction}} =
\operatorname{mean}\bigl[(x - f_\theta(\tilde{x}))^2\bigr].$$

Под `mean` здесь понимается среднее квадратов разностей, а не квадрат
средней разности. Затем усредняем ошибки изображений по test-набору.
Положительная разница «до − после» означает снижение MSE;
отрицательная означает, что модель ухудшила результат по этой метрике.

In [ ]:
y_noisy_pred = autoencoder.predict(x_test_noisy, batch_size=batch_size, verbose=0)
assert y_noisy_pred.shape == x_test.shape

noise_mse_per_image = np.mean(
    (x_test - x_test_noisy) ** 2, axis=(1, 2, 3)
)
reconstruction_mse_per_image = np.mean(
    (x_test - y_noisy_pred) ** 2, axis=(1, 2, 3)
)
noise_mse = float(noise_mse_per_image.mean())
reconstruction_mse = float(reconstruction_mse_per_image.mean())

print(f"Test MSE до восстановления: {noise_mse:.6f}")
print(f"Test MSE после восстановления: {reconstruction_mse:.6f}")
print(f"Разница MSE (до − после): {noise_mse - reconstruction_mse:.6f}")

## 5. Визуальная проверка

Показываем пять разных изображений, выбранных из всего test-набора
с фиксированным seed. В каждой строке: оригинал, шумный вход и
восстановление того же объекта. Цветовая шкала одинакова: `[0, 1]`.

In [ ]:
num_examples = 5
indices = plot_rng.choice(len(x_test), size=num_examples, replace=False)
fig, axes = plt.subplots(num_examples, 3, figsize=(8, 2 * num_examples))

for row, idx in enumerate(indices):
    images = (x_test[idx], x_test_noisy[idx], y_noisy_pred[idx])
    titles = (
        "Оригинал",
        f"Шум: MSE={noise_mse_per_image[idx]:.4f}",
        f"Восстановление: MSE={reconstruction_mse_per_image[idx]:.4f}",
    )
    for ax, image, title in zip(axes[row], images, titles):
        ax.imshow(image.squeeze(), cmap="gray", vmin=0, vmax=1)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
fig.tight_layout()
plt.show()

## 6. Интерпретация и ограничения

Метрики текущей версии ещё не получены. После запуска сравните MSE
шумного входа и восстановления на test-наборе, затем посмотрите
примеры: какие детали модель сохранила, а какие сгладила.

Из-за усреднения MSE модель может сглаживать детали изображения.
Снижение MSE не гарантирует сохранение всех визуально важных деталей.
Одна эпоха и один seed дают ограниченную картину качества.
